In [1]:
!pip install pandas torch transformers datasets scikit-learn


In [21]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [22]:
data = {
    "text": [
        "Payment failed twice",
        "Cannot login account",
        "Package damaged",
        "Refund pending",
        "Delivery delayed"
    ],
    "label": ["Billing", "Technical", "Delivery", "Billing", "Delivery"]
}

df = pd.DataFrame(data)

In [23]:
label2id = {label: idx for idx, label in enumerate(df['label'].unique())}
id2label = {idx: label for label, idx in label2id.items()}
df['label_id'] = df['label'].map(label2id)

In [24]:
dataset = Dataset.from_pandas(df)

In [25]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
from datasets import Dataset, DatasetDict # Ensure DatasetDict is imported

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)

# The error indicates 'dataset' is a DatasetDict. We need to split a specific split within it.
# Assuming the primary data is in a 'train' key within the DatasetDict.
if isinstance(dataset, DatasetDict) and 'train' in dataset:
    # Split the 'train' part of the DatasetDict into new 'train' and 'test' splits.
    split_result = dataset['train'].train_test_split(test_size=0.2)
    # Overwrite the 'dataset' variable with the new DatasetDict containing 'train' and 'test' splits.
    dataset = split_result
elif isinstance(dataset, Dataset):
    # If it was actually a single Dataset object, then this is the correct way to split it.
    # This path contradicts the error, but handles the theoretical case if the error was misleading.
    dataset = dataset.train_test_split(test_size=0.2)
else:
    raise TypeError("Unexpected type for 'dataset'. Expected Dataset or DatasetDict with 'train' split.")

# Remove the original 'label' column to avoid conflict with the new 'labels' column
# The 'labels' column already exists, so no renaming from 'label_id' is needed.
dataset['train'] = dataset['train'].remove_columns(['label'])
dataset['test'] = dataset['test'].remove_columns(['label'])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [37]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

In [38]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,1.129752
2,No log,1.143603
3,No log,1.150857


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=3, training_loss=1.0648310979207356, metrics={'train_runtime': 41.3762, 'train_samples_per_second': 0.073, 'train_steps_per_second': 0.073, 'total_flos': 397409283072.0, 'train_loss': 1.0648310979207356, 'epoch': 3.0})

In [39]:
trainer.save_model("./ticket_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [40]:
text = "Refund not processed"
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)
predicted_class = id2label[outputs.logits.argmax().item()]
print("Predicted Label:", predicted_class)

Predicted Label: Billing
